# BHI supplementary ECG processing and Figure 4

This notebook documents the study's EDF-to-ECG/IBI processing and historical NerMI and SAI/PAI workflows. NerMI and SAI/PAI are separately patented methods with different rights holders; their implementations are not included here. Licensing and analysis options are summarized in the repository README. The online SAI/PAI calculator warns that uploads are not encrypted; do not upload sensitive participant data. **Raw participant data are not supplied.** Some paths and dependencies require the original study environment.

## How to run from existing processed data

For Figure 4 alone, run the **Figure 4 setup**, **Load saved data**, and **Figure 4** cells below. Set `PROJECT_DIR` to your local study directory and leave `SOURCE_MODE = "processed"`. You do **not** need to run the upstream EDF processing cells. A combined saved CSV is also supported if it includes valence, motivation, and activation.

## Upstream processing record (optional, requires private original data)

The original EDF and feature-extraction code is preserved below. It requires the local EDF files, subject folders, and study-era dependencies. The NerMI extraction cell searches the `Interventi` and `Controlli` folders automatically. Subject folder names are used as participant identifiers, so keep generated files private. **Do not use Run All for raw processing without verifying paths and dependencies.** The exploratory figure-making cells unrelated to the manuscript have been removed. All saved execution outputs are cleared.


## Optional upstream processing from EDF files

The following cells document the original acquisition-to-feature workflow. They are not required when reproducing Figure 4 from saved processed data.


In [ ]:
import os
from pathlib import Path

DATA_DIR = Path(r"D:\Data\Projects\BHI\Data")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Study data directory not found: {DATA_DIR}")


In [ ]:
import sys
import site
import os
import re
import pandas as pd
import mne
from concurrent.futures import ThreadPoolExecutor, as_completed

# Setup environment and paths
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

current_dir = os.fspath(DATA_DIR)
groups = ['Interventi', 'Controlli']
subjects = [
    (group, name)
    for group in groups
    for group_dir in [os.path.join(current_dir, group)]
    if os.path.isdir(group_dir)
    for name in os.listdir(group_dir)
    if os.path.isdir(os.path.join(group_dir, name))
]
if not subjects:
    raise FileNotFoundError(f"No subject folders found under {current_dir}\\{{{','.join(groups)}}}")
participants = len(subjects)
trials = ['T1','T0']
tasks=['EMDR','script'] + ['oc'+ str(i) for i in range(1,4)] + ['oa'+ str(i) for i in range(1,4)]
# Threaded function to process each subject
def generate_physio(group,sub,trial,task):
    sub_save = os.path.join(current_dir, group, sub)
    raw_file = os.path.join(sub_save, f'{sub} {trial} {task}.edf')
    physio_file = os.path.join(sub_save, f'{sub} {trial} {task}.csv')

    if not os.path.exists(raw_file):
        return "Input EDF not found; skipping."
    if os.path.exists(physio_file):
        return "Existing output skipped."
    try:
        raw = mne.io.read_raw_edf(raw_file, preload=True)
        data, times = raw[:]
        data, times = raw.get_data(return_times=True)
        df = pd.DataFrame(data.T, columns=raw.ch_names)
        df["time"] = times
        df = df[["time"] + raw.ch_names]
        df.to_csv(physio_file, index=False, mode="x")
        return "ECG processed."
    except Exception as e:
        return "ECG processing failed; check private logs."

# Execute with threading
print(f"Starting regeneration using {min(32, len(subjects))} threads...")
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = [executor.submit(generate_physio, group, sub, trial, task) for task in tasks for trial in trials for group, sub in subjects]
    for i, future in enumerate(as_completed(futures), 1):
        print(f"{i}/{participants*2*len(tasks)}: {future.result()}")

print("✅ All done.")


## NerMI extraction from discovered subject folders

This cell searches the `Interventi` and `Controlli` data folders for subject subdirectories and processes the subjects it finds. It uses the subject folder names as participant identifiers.


In [ ]:
import os
import re
import pandas as pd
import nermi
from concurrent.futures import ThreadPoolExecutor, as_completed

current_dir = os.fspath(DATA_DIR)
trials = ['T1','T0']
tasks=['EMDR','script'] + ['oc'+ str(i) for i in range(1,4)] + ['oa'+ str(i) for i in range(1,4)]
groups=['Interventi','Controlli']
subjects = [
    (group, os.path.basename(sub_dir))
    for group in groups
    for group_dir in [os.path.join(current_dir, group)]
    if os.path.isdir(group_dir)
    for sub_dir in [os.path.join(group_dir, name) for name in os.listdir(group_dir)]
    if os.path.isdir(sub_dir)
]
if not subjects:
    raise FileNotFoundError(f"No subject folders found under {current_dir}\\{{{','.join(groups)}}}")
participants = len(subjects)
def generate_physio(group,sub,trial,task):
    try: 
        sub_save = os.path.join(current_dir, group, sub)
        pattern = re.compile(fr'^{re.escape(sub)}\s+{trial}$', re.IGNORECASE)
        candidates = [
            f for f in os.listdir(sub_save)
            if f.endswith(f"{task}.csv") and pattern.match(f.split(task)[0].strip())
        ]
        if candidates:
            physio_file = os.path.join(sub_save, candidates[0])
        else:
            return f"{sub} {trial} {task} file missing."
        save_dir = os.path.join("processed", group)
        os.makedirs(save_dir, exist_ok=True)
        save_file = os.path.join(save_dir, f'{sub}_{trial}_{task}_physio.csv')
        if os.path.exists(save_file):
            return f'{sub} {trial} {task} already run.'
        else:
            df=pd.read_csv(physio_file)
            frequency=round(1/abs(df.time[1]-df.time[2]))
            tdf=nermi.analysis(df.time,ECG=df['ECG-RF'],frequency=frequency,compute_emotions=True)
            tdf.to_csv(save_file, mode="x")
            return f'{sub} {trial} {task} success {frequency}.'
    except:
        return 'Processing failed; check private processing logs.'


print(f"Starting regeneration using {min(32, len(subjects))} threads...")
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = [executor.submit(generate_physio, group, sub, trial, task) for task in tasks for trial in trials for group, sub in subjects]
    for i, future in enumerate(as_completed(futures), 1):
        print(f"{i}/{participants*2*len(tasks)}: {future.result()}")


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import os
import re
import pandas as pd
from psychological_monitoring import spai

current_dir = os.path.abspath(r"processed")
group='Interventi'
Interventi = os.path.abspath(os.path.join(current_dir, group))
Interventisubs = pd.Series([x for x in os.listdir(Interventi)])
group='Controlli'
Controlli = os.path.abspath(os.path.join(current_dir, group))
Controllisubs = pd.Series([x for x in os.listdir(Controlli)])
x1=pd.DataFrame({'Name':Interventisubs})
x1['group']='Interventi'
x2=pd.DataFrame({'Name':Controllisubs})
x2['group']='Controlli'
subs=pd.concat([x1,x2])
include = ["oc1", "oc2", "script"]

subs = subs[subs["Name"].str.contains("|".join(include), case=False, na=False)]


df=pd.DataFrame()
for sub, group in subs.to_numpy()[:]:
    file_match = re.fullmatch(r"(.+)_t(\d+)_(emdr|script|oc[1-3]|oa[1-3])(?:_physio)?\.csv", sub, flags=re.I)
    if file_match is None:
        raise ValueError(f"Unexpected processed filename: {sub}")
    tdf=pd.read_csv(current_dir+fr'\{group}\{sub}')
    rdf=spai.runall(tdf.timestamps,report=False)
    tdf=pd.concat([tdf,rdf],axis=1)
    tdf['participant']=file_match.group(1)
    tdf['condition']=group
    tdf['trial']=file_match.group(2)
    tdf['task']=file_match.group(3).lower()
    df=pd.concat([df,tdf[2:]],axis=0)
df=df.reset_index(drop=True)
df

In [ ]:
# Preprocess SAI/PAI and build plotting/export tables
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# SETTINGS
# =========================================================

K_MAD = 3
PHASE_ORDER = ["oc1", "script", "oc2"]

required_cols = ["participant", "condition", "trial", "task", "timestamps"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"df is missing required columns: {missing}")

sai_source = "SAI_N" if "SAI_N" in df.columns else "SAI"
pai_source = "PAI_N" if "PAI_N" in df.columns else "PAI"
if sai_source not in df.columns or pai_source not in df.columns:
    raise ValueError("df must contain SAI_N/PAI_N or SAI/PAI columns")

workdf = df.copy()
workdf = workdf[workdf["task"].astype(str).str.lower().isin(PHASE_ORDER)].copy()

workdf["timestamps"] = pd.to_numeric(workdf["timestamps"], errors="coerce")
workdf["SAI"] = pd.to_numeric(workdf[sai_source], errors="coerce")
workdf["PAI"] = pd.to_numeric(workdf[pai_source], errors="coerce")
workdf["task"] = workdf["task"].astype(str).str.lower()
workdf["trial"] = workdf["trial"].astype(str)
workdf["participant"] = workdf["participant"].astype(str).str.strip()
workdf["condition"] = workdf["condition"].astype(str).str.strip()

# =========================================================
# PREPROCESSING
# =========================================================

def mad_based_mask(x, k=3):
    """Keep finite values within k * MAD of the segment median."""
    x = pd.Series(x).astype(float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))

    if np.isnan(mad) or mad == 0:
        return ~x.isna()

    return (~x.isna()) & (np.abs(x - med) <= k * mad)


def preprocess_segment(g, k=3):
    """Clean one participant x condition x trial x phase segment."""
    g = g.sort_values("timestamps").copy()

    sai_mask = mad_based_mask(g["SAI"], k=k)
    pai_mask = mad_based_mask(g["PAI"], k=k)

    g["SAI_clean"] = g["SAI"].where(sai_mask)
    g["PAI_clean"] = g["PAI"].where(pai_mask)

    return g


segment_keys = ["participant", "condition", "trial", "task"]
cleaned = (
    workdf.groupby(segment_keys, group_keys=False)
    .apply(lambda g: preprocess_segment(g, k=K_MAD))
    .reset_index(drop=True)
)

# =========================================================
# SEGMENT MEDIANS AND PRE/POST TABLES FOR FIGURES
# =========================================================

segment_summary = (
    cleaned.groupby(segment_keys)
    .agg(
        SAI_median=("SAI_clean", "median"),
        PAI_median=("PAI_clean", "median"),
        n_SAI=("SAI_clean", lambda x: x.notna().sum()),
        n_PAI=("PAI_clean", lambda x: x.notna().sum())
    )
    .reset_index()
)

wide = segment_summary.pivot_table(
    index=["participant", "condition", "trial"],
    columns="task",
    values=["SAI_median", "PAI_median", "n_SAI", "n_PAI"],
    aggfunc="first"
)
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["SAI_reactivity"] = wide["SAI_median_script"] - wide["SAI_median_oc1"]
wide["PAI_reactivity"] = wide["PAI_median_script"] - wide["PAI_median_oc1"]
wide["SAI_recovery"] = wide["SAI_median_oc2"] - wide["SAI_median_script"]
wide["PAI_recovery"] = wide["PAI_median_oc2"] - wide["PAI_median_script"]


def normalize_trial_label(x):
    s = str(x).strip().upper()
    if s in ["0", "T0"]:
        return "T0"
    if s in ["1", "T1"]:
        return "T1"
    return s


wide["trial_norm"] = wide["trial"].apply(normalize_trial_label)
prepost = wide[wide["trial_norm"].isin(["T0", "T1"])].copy()

prepost_wide = prepost.pivot_table(
    index=["participant", "condition"],
    columns="trial_norm",
    values=[
        "SAI_median_oc1", "SAI_median_script", "SAI_median_oc2",
        "PAI_median_oc1", "PAI_median_script", "PAI_median_oc2",
        "SAI_reactivity", "PAI_reactivity",
        "SAI_recovery", "PAI_recovery"
    ],
    aggfunc="first"
)
prepost_wide.columns = [f"{var}_{tp}" for var, tp in prepost_wide.columns]
prepost_wide = prepost_wide.reset_index()

change_vars = [
    "SAI_median_oc1", "SAI_median_script", "SAI_median_oc2",
    "PAI_median_oc1", "PAI_median_script", "PAI_median_oc2",
    "SAI_reactivity", "PAI_reactivity",
    "SAI_recovery", "PAI_recovery"
]

for v in change_vars:
    c0 = f"{v}_T0"
    c1 = f"{v}_T1"
    if c0 in prepost_wide.columns and c1 in prepost_wide.columns:
        prepost_wide[f"{v}_change"] = prepost_wide[c1] - prepost_wide[c0]

# =========================================================
# OPTIONAL PARTICIPANT-LEVEL FIGURE HELPERS
# =========================================================

def plot_participant_timeseries(cleaned_df, participant, condition=None, trial=None,
                                use_clean=True, save_dir=None, show=True):
    g = cleaned_df[cleaned_df["participant"] == participant].copy()

    if condition is not None:
        g = g[g["condition"] == condition]
    if trial is not None:
        g = g[g["trial"].astype(str) == str(trial)]

    if g.empty:
        print("No data found for requested participant and selections")
        return

    value_cols = ("SAI_clean", "PAI_clean") if use_clean else ("SAI", "PAI")
    g["task"] = pd.Categorical(g["task"], categories=PHASE_ORDER, ordered=True)
    g = g.sort_values(["trial", "task", "timestamps"])

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

    for ax, ycol, title in zip(axes, value_cols, ["SAI dynamics", "PAI dynamics"]):
        for phase in PHASE_ORDER:
            gt = g[g["task"] == phase].copy()
            if gt.empty:
                continue
            ax.plot(gt["timestamps"], gt[ycol], label=phase)
        ax.set_title(f"{title} | condition={condition} | trial={trial}")
        ax.set_xlabel("timestamps")
        ax.set_ylabel(ycol)
        ax.legend()

    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{participant}_{condition}_{trial}_timeseries.png"
        with open(os.path.join(save_dir, fname), "xb") as image_file:
            plt.savefig(image_file, dpi=200, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()


def plot_all_participants(cleaned_df, save_dir=None, use_clean=True):
    keys = cleaned_df[["participant", "condition", "trial"]].drop_duplicates()
    for _, row in keys.iterrows():
        plot_participant_timeseries(
            cleaned_df=cleaned_df,
            participant=row["participant"],
            condition=row["condition"],
            trial=row["trial"],
            use_clean=use_clean,
            save_dir=save_dir,
            show=False
        )

print("cleaned rows:", len(cleaned))
print("segment_summary rows:", len(segment_summary))
print("prepost_wide rows:", len(prepost_wide))


In [ ]:
clinical_path = DATA_DIR / "clinical" / "participant_data_deidentified.xlsx"

clinical_df = pd.read_excel(clinical_path)
clinical_df = clinical_df.rename(columns={
    "PZ": "participant",
    "Group": "group_code"
})

clinical_df["participant"] = clinical_df["participant"].astype(str).str.strip()

# -----------------------------
# 1. Clean participant key
# -----------------------------
clinical_df = clinical_df.rename(columns={"PZ": "participant", "Group": "group_code"}).copy()
clinical_df["participant"] = clinical_df["participant"].astype(str).str.strip()
prepost_wide["participant"] = prepost_wide["participant"].astype(str).str.strip()

# -----------------------------
# 2. CAPS change
# -----------------------------
clinical_df["CAPS_T0"] = pd.to_numeric(clinical_df["CAPST0TOT"], errors="coerce")
clinical_df["CAPS_T1"] = pd.to_numeric(clinical_df["CAPST1TOT"], errors="coerce")
clinical_df["CAPS_change"] = clinical_df["CAPS_T1"] - clinical_df["CAPS_T0"]



## Optional processed-series exports

These cells use objects prepared above and write functional CSV inputs for the R analysis. They are included for methodological transparency, not as prerequisites for the cached Figure 4 route.


In [ ]:
# assumes you already created `cleaned`
# columns needed:
# participant, condition, trial, task, timestamps, SAI_clean, PAI_clean

export_cols = [
    "participant", "condition", "trial", "task",
    "timestamps", "SAI_clean", "PAI_clean"
]

out = cleaned[export_cols].copy()

out["participant"] = out["participant"].astype(str).str.strip()
out["condition"] = out["condition"].astype(str).str.strip()
out["trial"] = out["trial"].astype(str).str.upper().replace({"0": "T0", "1": "T1"})
out["task"] = out["task"].astype(str).str.lower()

out = out[out["task"].isin(["oc1", "script", "oc2"])].copy()

out.to_csv(r"outputs/functional_cleaned_long.csv", index=False, mode="x")
print("saved:", r"outputs/functional_cleaned_long.csv")



In [ ]:
import os
import pandas as pd
import numpy as np

out = df.copy()

out["participant"] = out["participant"].astype(str).str.strip()
out["condition"] = out["condition"].astype(str).str.strip()
out["task"] = out["task"].astype(str).str.lower().str.strip()
out["trial"] = out["trial"].astype(str).str.upper().str.strip()

out["trial"] = out["trial"].replace({
    "0": "T0",
    "1": "T1"
})

out = out[out["task"].isin(["oc1", "script", "oc2"])].copy()

keep_cols = [
    "participant",
    "condition",
    "trial",
    "task",
    "timestamps",
    "NERMI_Valence",
    "NERMI_Motivation"
]

out = out[keep_cols].copy()

for col in ["timestamps", "NERMI_Valence", "NERMI_Motivation"]:
    out[col] = pd.to_numeric(out[col], errors="coerce")

save_path = r"outputs/functional_cleaned_long_nermi.csv"
out.to_csv(save_path, index=False, mode="x")

print(save_path)
print(out.head())
print(out.shape)

## Figure 4 from previously saved processing results

Run the next three code cells in order. This is the independent entry point for the figure; it does not depend on executing the original EDF processing cells above. The figure compares NerMI valence, motivation, and activation correlations. Activation is retained for comparison, but is not an outcome in the ECG-only analysis.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

PROJECT_DIR = Path(r"D:\Data\Projects\BHI\Data")
SOURCE_MODE = "processed"  # "processed" or "combined"
PROCESSED_DIR = PROJECT_DIR / "ECG_Processed"
COMBINED_CSV = PROJECT_DIR / "functional_full_nermi.csv"  # Requires all three NerMI dimensions
FIGURE_PATH = PROJECT_DIR / "Figures" / "Figure_4_correlations.png"
GROUP_COLORS = {"TAU": "#4C78A8", "EMDR": "#F58518"}


In [ ]:
REQUIRED = {"participant", "condition", "trial", "task", "NERMI_Valence", "NERMI_Motivation", "NERMI_Activation"}

def load_processed(processed_dir):
    frames = []
    for folder, condition in (("Interventi", "EMDR"), ("Controlli", "TAU")):
        group_dir = processed_dir / folder
        if not group_dir.is_dir():
            raise FileNotFoundError(f"Missing processed-data directory: {group_dir}")
        files = sorted(group_dir.glob("*_physio.csv"))
        if not files:
            raise FileNotFoundError(f"No *_physio.csv files found in {group_dir}")
        for path in files:
            # Saved filename convention: {participant}_{T0|T1}_{phase}_physio.csv.
            match = re.fullmatch(r"(.+?)_(t[01])_(oc1|script|oc2)_physio", path.stem, flags=re.I)
            if match is None:
                continue
            participant, trial, phase = match.groups()
            frame = pd.read_csv(path)
            required_signals = {"NERMI_Valence", "NERMI_Motivation", "NERMI_Activation"}
            missing = required_signals - set(frame.columns)
            if missing:
                raise ValueError(f"{path.name} lacks {sorted(missing)}; regenerate or select a complete saved dataset")
            frame = frame[list(sorted(required_signals))].copy()
            frame["participant"] = participant
            frame["condition"] = condition
            frame["trial"] = trial.upper()
            frame["task"] = phase.lower()
            frames.append(frame)
    if not frames:
        raise ValueError("No processed files matched {participant}_{T0|T1}_{oc1|script|oc2}_physio.csv")
    return pd.concat(frames, ignore_index=True)

if SOURCE_MODE == "processed":
    df = load_processed(PROCESSED_DIR)
elif SOURCE_MODE == "combined":
    if not COMBINED_CSV.is_file():
        raise FileNotFoundError(f"Combined file not found: {COMBINED_CSV}")
    df = pd.read_csv(COMBINED_CSV)
else:
    raise ValueError('SOURCE_MODE must be "processed" or "combined"')

missing = REQUIRED - set(df.columns)
if missing:
    raise ValueError(f"Loaded data lacks {sorted(missing)}; Figure 4 needs all three NerMI dimensions")
df["condition"] = df["condition"].astype(str).replace({"Interventi": "EMDR", "Controlli": "TAU"})
df["task"] = df["task"].astype(str).str.lower()
df["participant"] = df["participant"].astype(str)
for field in ("NERMI_Valence", "NERMI_Motivation", "NERMI_Activation"):
    df[field] = pd.to_numeric(df[field], errors="coerce")
print(f"Loaded {len(df):,} samples for {df['participant'].nunique()} participants.")


In [ ]:
# Figure 4: each participant contributes one script-phase absolute correlation per pair.
# Matches the original figure's across-visit pooling; no T0/T1 split.
pairs = [("NERMI_Valence", "NERMI_Motivation", "Valence-Motivation"),
         ("NERMI_Valence", "NERMI_Activation", "Valence-Activation"),
         ("NERMI_Motivation", "NERMI_Activation", "Motivation-Activation")]
rows = []
script = df.loc[df["task"].eq("script")]
for (condition, participant), frame in script.groupby(["condition", "participant"], sort=True):
    for left, right, label in pairs:
        xy = frame[[left, right]].dropna()
        if len(xy) < 5 or xy[left].std() == 0 or xy[right].std() == 0:
            corr = np.nan
        else:
            corr = abs(float(pearsonr(xy[left], xy[right]).statistic))
        rows.append({"Condition": condition, "Correlation": label, "Absolute correlation": corr})
figure_data = pd.DataFrame(rows)
if figure_data.empty or figure_data["Absolute correlation"].notna().sum() == 0:
    raise ValueError("No valid script-phase correlations: check saved files and signal columns")

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=figure_data, x="Correlation", y="Absolute correlation",
               hue="Condition", hue_order=["TAU", "EMDR"], palette=GROUP_COLORS,
               split=True, cut=0, ax=ax)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylim(0, 1.05)
ax.set_title("Absolute correlations by condition during script exposure")
fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
with FIGURE_PATH.open("xb") as figure_file:
    fig.savefig(figure_file, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved Figure 4 to {FIGURE_PATH}")


## Scope and disclosure

This notebook provides methods documentation, not participant-level data. `BHI.r` and `BHI_NerMI.r` contain other manuscript analyses. Raw processing requires the original study environment; Figure 4 requires saved data containing all three NerMI dimensions.
